## Part 1 — Setup & Environment

In [ ]:
# Standard imports and reproducibility
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from typing import List, Tuple

# Reproducible results
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Plot settings
sns.set(style='whitegrid')

# --- Configuration ---
GRID_SIZE = 5
N_STATES = GRID_SIZE * GRID_SIZE
N_ACTIONS = 4  # Up, Down, Left, Right
GAMMA = 0.9    # Discount factor

# Actions mapping
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
ACTIONS = [UP, DOWN, LEFT, RIGHT]
ACTION_NAMES = {0: '↑', 1: '↓', 2: '←', 3: '→'}

In [ ]:
class GridWorld:
    def __init__(self, size: int = 5):
        self.size = size
        self.n_states = size * size
        self.goal_state = self.n_states - 1  # bottom-right cell

    def get_next_state(self, state: int, action: int) -> int:
        row, col = divmod(state, self.size)
        if action == UP:
            row = max(row - 1, 0)
        elif action == DOWN:
            row = min(row + 1, self.size - 1)
        elif action == LEFT:
            col = max(col - 1, 0)
        elif action == RIGHT:
            col = min(col + 1, self.size - 1)
        return row * self.size + col

    def get_features(self, state: int) -> np.ndarray:
        # One-hot features for each state
        f = np.zeros(self.n_states)
        f[state] = 1.0
        return f

    def get_transition_probs(self) -> np.ndarray:
        # Deterministic transitions P[s, a, s']
        P = np.zeros((self.n_states, N_ACTIONS, self.n_states))
        for s in range(self.n_states):
            for a in ACTIONS:
                next_s = self.get_next_state(s, a)
                P[s, a, next_s] = 1.0
        return P


env = GridWorld(GRID_SIZE)
print(f"GridWorld {GRID_SIZE}x{GRID_SIZE} initialized with {env.n_states} states.")

## Part 2 — The Expert (Value Iteration)

In [ ]:
def value_iteration(env: GridWorld, reward_vector: np.ndarray, theta: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """Return (policy, V) for the given reward vector using value iteration."""
    V = np.zeros(env.n_states)
    P = env.get_transition_probs()

    while True:
        delta = 0.0
        for s in range(env.n_states):
            v = V[s]
            q_values = np.zeros(N_ACTIONS)
            for a in ACTIONS:
                q_values[a] = reward_vector[s] + GAMMA * np.dot(P[s, a], V)
            V[s] = np.max(q_values)
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break

    # Deterministic greedy policy
    policy = np.zeros(env.n_states, dtype=int)
    for s in range(env.n_states):
        q_values = np.zeros(N_ACTIONS)
        for a in ACTIONS:
            q_values[a] = reward_vector[s] + GAMMA * np.dot(P[s, a], V)
        policy[s] = np.argmax(q_values)

    return policy, V


# Ground truth reward: -1 step cost, +10 at goal, -10 at a puddle in the middle
ground_truth_rewards = np.full(env.n_states, -1.0)
ground_truth_rewards[env.goal_state] = 10.0
puddle_state = 12  # central cell in 5x5
ground_truth_rewards[puddle_state] = -10.0

expert_policy, expert_values = value_iteration(env, ground_truth_rewards)

# Visualization helper

def plot_grid(values: np.ndarray, title: str = "Grid", cmap: str = "RdYlGn", fmt: str = ".1f"):
    grid = values.reshape((GRID_SIZE, GRID_SIZE))
    plt.figure(figsize=(5, 4))
    sns.heatmap(grid, annot=True, fmt=fmt, cmap=cmap, cbar=True)
    plt.title(title)
    plt.show()

print("Expert training complete.")
plot_grid(ground_truth_rewards, "Ground Truth Reward Function (Hidden from IRL)")
plot_grid(expert_values, "Expert Value Function")

## Part 3 — Generating Expert Demonstrations

In [ ]:
def generate_trajectories(env: GridWorld, policy: np.ndarray, n_trajectories: int = 50, max_steps: int = 20) -> List[List[int]]:
    trajectories = []
    for _ in range(n_trajectories):
        start_state = np.random.randint(0, env.n_states - 1)  # avoid starting at goal
        traj = []
        state = start_state
        for _ in range(max_steps):
            traj.append(state)
            if state == env.goal_state:
                break
            action = policy[state]
            state = env.get_next_state(state, action)
        trajectories.append(traj)
    return trajectories

expert_trajs = generate_trajectories(env, expert_policy, n_trajectories=80, max_steps=20)
print(f"Generated {len(expert_trajs)} expert trajectories. Sample: {expert_trajs[0]}")

## Part 4 — Maximum Entropy IRL (Feature Matching)

We use a simple gradient-ascent method to match feature expectations. For this demo, features are one-hot per state, so the learned reward is linear in these features (i.e., one weight per state).

In [ ]:
def compute_expert_feature_expectations(env: GridWorld, trajectories: List[List[int]]) -> np.ndarray:
    feat_exp = np.zeros(env.n_states)
    for traj in trajectories:
        for s in traj:
            feat_exp += env.get_features(s)
    feat_exp /= len(trajectories)
    return feat_exp


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)


def max_ent_irl(env: GridWorld, expert_trajs: List[List[int]], lr: float = 0.1, n_iters: int = 50, horizon: int = 20, temp: float = 1.0):
    """Return recovered weights theta and loss history."""
    expert_feat_exp = compute_expert_feature_expectations(env, expert_trajs)

    # Initialize weights
    theta = np.random.uniform(low=-0.1, high=0.1, size=env.n_states)
    P_trans = env.get_transition_probs()

    loss_history = []

    for it in range(n_iters):
        # Rewards are linear: R[s] = theta[s] because phi is one-hot
        rewards = theta.copy()

        # Solve MDP (we use standard VI to get approximate state values)
        _, state_values = value_iteration(env, rewards)

        # Build soft policy pi(a|s) using Q(s,a) = R(s) + gamma * sum_s' P[s,a,s'] * V(s')
        policy_prob = np.zeros((env.n_states, N_ACTIONS))
        for s in range(env.n_states):
            qs = np.zeros(N_ACTIONS)
            for a in ACTIONS:
                qs[a] = rewards[s] + GAMMA * np.dot(P_trans[s, a], state_values)
            policy_prob[s] = softmax(qs / max(temp, 1e-8))

        # Estimate state visitation frequencies via forward propagation
        p_start = np.ones(env.n_states) / env.n_states
        curr_p = p_start.copy()
        state_freq = np.copy(curr_p)

        for t in range(horizon):
            next_p = np.zeros(env.n_states)
            for s in range(env.n_states):
                if curr_p[s] <= 0:
                    continue
                for a in ACTIONS:
                    prob_a = policy_prob[s, a]
                    next_s = env.get_next_state(s, a)
                    next_p[next_s] += curr_p[s] * prob_a
            curr_p = next_p
            state_freq += (GAMMA ** (t + 1)) * curr_p

        # Learner feature expectations from state_freq
        learner_feat_exp = np.zeros(env.n_states)
        for s in range(env.n_states):
            learner_feat_exp += state_freq[s] * env.get_features(s)

        gradient = expert_feat_exp - learner_feat_exp
        theta += lr * gradient

        # Regularize / scale to avoid divergence (heuristic)
        theta = theta / (np.linalg.norm(theta) + 1e-8)

        loss = np.linalg.norm(expert_feat_exp - learner_feat_exp)
        loss_history.append(loss)

        if (it + 1) % 10 == 0 or it == 0:
            print(f"Iter {it+1}/{n_iters} — feature error: {loss:.6f}")

    return theta, loss_history


# Run IRL
print("Running MaxEnt IRL (this may take a few seconds)...")
recovered_weights, loss_history = max_ent_irl(env, expert_trajs, lr=0.5, n_iters=100, horizon=15, temp=1.0)
recovered_rewards = recovered_weights.copy()
print("IRL finished.")

## Part 5 — Analysis & Visualization

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(loss_history, lw=2)
plt.xlabel('Iteration')
plt.ylabel('||Expert_Features - Learner_Features||')
plt.title('IRL Feature Matching Error')
plt.grid(True)
plt.show()

# Compare Ground Truth vs Recovered
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) Ground Truth
grid_gt = ground_truth_rewards.reshape((GRID_SIZE, GRID_SIZE))
sns.heatmap(grid_gt, ax=axes[0], annot=True, fmt='.1f', cmap='RdYlGn')
axes[0].set_title('Ground Truth Reward')

# 2) Recovered Reward (Normalized for display)
rec_norm = (recovered_rewards - np.mean(recovered_rewards)) / (np.std(recovered_rewards) + 1e-8)
grid_rec = rec_norm.reshape((GRID_SIZE, GRID_SIZE))
sns.heatmap(grid_rec, ax=axes[1], annot=False, cmap='RdYlGn')
axes[1].set_title('Recovered Reward (Normalized)')

# 3) Policy from Recovered Reward
new_policy, _ = value_iteration(env, recovered_rewards)
policy_grid = new_policy.reshape((GRID_SIZE, GRID_SIZE))

axes[2].matshow(np.zeros((GRID_SIZE, GRID_SIZE)), cmap='Greys')
for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        action = policy_grid[i, j]
        txt = ACTION_NAMES.get(action, '?')
        axes[2].text(j, i, txt, ha='center', va='center', fontsize=18, color='black')
axes[2].set_title('Policy from Recovered Reward')

plt.tight_layout()
plt.show()

# Visualize a few expert trajectories on top of the grid
plt.figure(figsize=(5, 5))
G = np.zeros((GRID_SIZE, GRID_SIZE))
ax = sns.heatmap(G, cmap='Greys', cbar=False, linecolor='lightgray', linewidths=0.5)
for k in range(6):
    traj = expert_trajs[k]
    coords = [(s % GRID_SIZE, s // GRID_SIZE) for s in traj]
    xs = [c[0] + 0.5 for c in coords]
    ys = [c[1] + 0.5 for c in coords]
    plt.plot(xs, ys, marker='o', label=f'traj {k+1}')

plt.gca().invert_yaxis()
plt.title('Sample Expert Trajectories')
plt.legend(loc='upper left')
plt.show()

## Interpretation & Notes

- IRL recovers structure of the reward function (high at the goal, low at the puddle) but not the exact scale — this is expected.
- The recovered policy should be qualitatively similar to the expert's behavior.

If you want, I can run the notebook end-to-end and fix any runtime issues, or add: 
- richer feature sets (coordinates, distances),
- more robust MaxEnt implementations (exact forward-backward soft value iteration), or
- code to save and load recovered rewards.